# Exploration du dataset Olist (Phase 1, étape 1)

**Objectif :** comprendre chaque fichier CSV avant de construire Bronze, Silver et Gold.

Pour chaque fichier, on répond à 4 questions :
1. **Une ligne = quoi ?** (le grain)
2. **Quelle est la clé ?** (la colonne qui identifie une ligne)
3. **À quelles autres tables est-il relié ?**
4. **Quelles anomalies de qualité contient-il ?**

**Plan du notebook**

| Partie | Fichier |
|---|---|
| 0 | Préparation |
| 1 | `olist_orders_dataset.csv` |
| 2 | `olist_customers_dataset.csv` |
| 3 | `olist_order_items_dataset.csv` |
| 4 | `olist_order_payments_dataset.csv` |
| 5 | `olist_order_reviews_dataset.csv` |
| 6 | `olist_products_dataset.csv` + `product_category_name_translation.csv` |
| 7 | `olist_sellers_dataset.csv` |
| 8 | `olist_geolocation_dataset.csv` |
| 9 | Relations entre les tables |
| 10 | Synthèse : règles de qualité pour Silver |

**Comment travailler :** exécute les cellules avec `Maj+Entrée`, lis chaque résultat, puis remplis les cellules « À toi » avec tes propres mots.

## 0. Préparation

On définit les chemins des fichiers et deux petites fonctions :
- `show(sql)` : exécute une requête SQL DuckDB et affiche le résultat sous forme de tableau
- `profil(fichier)` : affiche, pour chaque colonne, son type, le nombre de valeurs vides et de valeurs distinctes

In [2]:
%pip install duckdb 
import duckdb
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Remonter jusqu'au dossier du projet pour trouver data/raw, peu importe le dossier courant.
candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((path for path in candidates if (path / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Impossible de trouver le dossier data/raw depuis le dossier courant.")
RAW = ROOT / "data" / "raw"


def chemin(nom):
    return (RAW / nom).as_posix()


CUSTOMERS = chemin("olist_customers_dataset.csv")
GEO = chemin("olist_geolocation_dataset.csv")
ITEMS = chemin("olist_order_items_dataset.csv")
PAYMENTS = chemin("olist_order_payments_dataset.csv")
REVIEWS = chemin("olist_order_reviews_dataset.csv")
ORDERS = chemin("olist_orders_dataset.csv")
PRODUCTS = chemin("olist_products_dataset.csv")
SELLERS = chemin("olist_sellers_dataset.csv")
CATEG = chemin("product_category_name_translation.csv")

# Base en mémoire : rien n'est écrit sur le disque
con = duckdb.connect()


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Vérifier que les 9 fichiers sont bien là
fichiers = [CUSTOMERS, GEO, ITEMS, PAYMENTS, REVIEWS, ORDERS, PRODUCTS, SELLERS, CATEG]
manquants = [f for f in fichiers if not Path(f).exists()]
assert not manquants, f"Fichiers introuvables : {manquants}. Vérifie le dossier {RAW}"
print("OK : les 9 fichiers sont présents dans", RAW)

OK : les 9 fichiers sont présents dans c:\Users\wiki\Desktop\olist-azure-lakehouse\data\raw


In [4]:
def show(sql):
    """Exécute une requête SQL et affiche le résultat sous forme de tableau."""
    display(con.sql(sql).df())


def profil(path):
    """Pour chaque colonne : type, valeurs vides, valeurs distinctes, % de vides."""
    cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").fetchall()
    n = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchall()[0][0]
    parts = []
    for c in cols:
        nom, typ = c[0], c[1]
        parts.append(
            f"SELECT '{nom}' AS colonne, '{typ}' AS type, "
            f'COUNT(*) - COUNT("{nom}") AS vides, '
            f'COUNT(DISTINCT "{nom}") AS distincts '
            f"FROM read_csv_auto('{path}')"
        )
    df = con.sql(" UNION ALL ".join(parts)).df()
    df["pct_vides"] = (100 * df["vides"] / n).round(2)
    print(f"{n} lignes, {len(cols)} colonnes")
    display(df)

## 1. `olist_orders_dataset.csv` : les commandes

**Rôle :** c'est la table **centrale** (le « hub ») : tout le reste se relie à elle par `order_id`.

**À retenir :** une ligne = **une commande**. Clé : `order_id`.

**Ce qu'on cherche :** les statuts possibles, les dates du parcours d'une commande, et les incohérences entre statut et dates.

In [11]:
show(f"SELECT * FROM read_csv_auto('{ORDERS}') LIMIT 5")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [12]:
profil(ORDERS)

99441 lignes, 8 colonnes


,colonne,type,vides,distincts,pct_vides
0,order_id,VARCHAR,0,99441,0.00
1,customer_id,VARCHAR,0,99441,0.00
2,order_status,VARCHAR,0,8,0.00
3,order_purchase_timestamp,TIMESTAMP,0,98875,0.00
4,order_approved_at,TIMESTAMP,160,90733,0.16
5,order_delivered_carrier_date,TIMESTAMP,1783,81018,1.79
6,order_delivered_customer_date,TIMESTAMP,2965,95664,2.98
7,order_estimated_delivery_date,TIMESTAMP,0,459,0.00


**À toi :** quelles colonnes ont des valeurs vides ? Pourquoi est-ce normal pour certaines d'entre elles ?

### 1.1 Les statuts

Parcours normal : `created` → `approved` → `invoiced` → `processing` → `shipped` → `delivered`.
Sorties possibles : `canceled` ou `unavailable`.

In [13]:
show(f"""
    SELECT order_status,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_csv_auto('{ORDERS}')
    GROUP BY order_status
    ORDER BY n DESC
""")

,order_status,n,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


### 1.2 Quelles dates sont remplies selon le statut ?

Les dates devraient se remplir **dans l'ordre du parcours** : approbation, puis transporteur, puis livraison.

In [14]:
show(f"""
    SELECT order_status,
           COUNT(*) AS total,
           COUNT(order_approved_at) AS avec_approbation,
           COUNT(order_delivered_carrier_date) AS avec_date_transporteur,
           COUNT(order_delivered_customer_date) AS avec_date_livraison
    FROM read_csv_auto('{ORDERS}')
    GROUP BY order_status
    ORDER BY total DESC
""")

,order_status,total,avec_approbation,avec_date_transporteur,avec_date_livraison
0,delivered,96478,96464,96476,96470
1,shipped,1107,1107,1107,0
2,canceled,625,484,75,6
3,unavailable,609,609,0,0
4,invoiced,314,314,0,0
5,processing,301,301,0,0
6,created,5,0,0,0
7,approved,2,2,0,0


### 1.3 Les incohérences entre statut et dates

On compte d'un coup toutes les règles violées. **Ce sont les premières règles de ta quarantaine.**

In [15]:
show(f"""
    SELECT
        COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL) AS delivered_sans_livraison,
        COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_approved_at IS NULL) AS delivered_sans_approbation,
        COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_carrier_date IS NULL) AS delivered_sans_transporteur,
        COUNT(*) FILTER (WHERE order_status = 'canceled' AND order_delivered_customer_date IS NOT NULL) AS canceled_avec_livraison,
        COUNT(*) FILTER (WHERE order_delivered_carrier_date < order_approved_at) AS transporteur_avant_approbation,
        COUNT(*) FILTER (WHERE order_delivered_customer_date < order_delivered_carrier_date) AS livraison_avant_transporteur,
        COUNT(*) FILTER (WHERE order_approved_at < order_purchase_timestamp) AS approbation_avant_achat,
        COUNT(*) FILTER (WHERE order_delivered_customer_date < order_purchase_timestamp) AS livraison_avant_achat
    FROM read_csv_auto('{ORDERS}')
""")

,delivered_sans_livraison,delivered_sans_approbation,delivered_sans_transporteur,canceled_avec_livraison,transporteur_avant_approbation,livraison_avant_transporteur,approbation_avant_achat,livraison_avant_achat
0,8,14,2,6,1359,23,0,0


Les commandes `delivered` **sans date de livraison** (une vraie contradiction) :

In [16]:
show(f"""
    SELECT order_id, order_status, order_purchase_timestamp, order_approved_at,
           order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
""")

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
1,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
2,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
3,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
4,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
5,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
6,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
7,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


### 1.4 Zoom : la remise au transporteur avant l'approbation

Un colis ne devrait pas partir avant que la commande soit approuvée. De combien est l'écart ?

In [17]:
show(f"""
    SELECT COUNT(*) AS nb,
           ROUND(AVG(date_diff('minute', order_delivered_carrier_date, order_approved_at)), 1) AS ecart_moyen_min,
           MEDIAN(date_diff('minute', order_delivered_carrier_date, order_approved_at)) AS ecart_median_min,
           MAX(date_diff('minute', order_delivered_carrier_date, order_approved_at)) AS ecart_max_min
    FROM read_csv_auto('{ORDERS}')
    WHERE order_delivered_carrier_date < order_approved_at
""")

,nb,ecart_moyen_min,ecart_median_min,ecart_max_min
0,1359,1484.8,1030.0,246555


In [18]:
# Ces commandes, par statut
show(f"""
    SELECT order_status, COUNT(*) AS n
    FROM read_csv_auto('{ORDERS}')
    WHERE order_delivered_carrier_date < order_approved_at
    GROUP BY order_status
    ORDER BY n DESC
""")

,order_status,n
0,delivered,1350
1,shipped,9


In [19]:
# Les 10 plus grands écarts
show(f"""
    SELECT order_id, order_status, order_approved_at, order_delivered_carrier_date,
           date_diff('day', order_delivered_carrier_date, order_approved_at) AS ecart_jours
    FROM read_csv_auto('{ORDERS}')
    WHERE order_delivered_carrier_date < order_approved_at
    ORDER BY ecart_jours DESC
    LIMIT 10
""")

,order_id,order_status,order_approved_at,order_delivered_carrier_date,ecart_jours
0,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:50:22,2018-01-26 13:35:00,171
1,77ca435b03fbf991e5027e3776e37885,delivered,2017-09-13 21:56:08,2017-09-04 20:50:00,9
2,cf72398d0690f841271b695bbfda82d2,delivered,2017-09-13 22:04:39,2017-09-04 20:12:41,9
3,1378f9601350615613cc8832d6789c5d,delivered,2017-09-13 22:03:51,2017-09-04 18:07:55,9
4,0184d4ddb259e1a4cfc2871888cf97b8,delivered,2017-09-13 22:17:15,2017-09-04 14:05:50,9
5,1fab4ac9d85079b3da72a11475ae1685,delivered,2017-09-13 22:06:11,2017-09-04 13:10:23,9
6,8554cb37f7158cb0b082a841d24a4589,delivered,2017-09-13 21:58:04,2017-09-04 19:12:19,9
7,bc4854efd86d9f42140c951c595d20c1,delivered,2017-09-13 22:00:51,2017-09-04 20:49:57,9
8,40de47dfa620d667117e4a6067b6e1ec,delivered,2017-09-13 21:58:38,2017-09-04 20:36:58,9
9,c3b8c17ee15e0e798c2e178b7d4c7f42,delivered,2017-09-13 22:08:55,2017-09-05 15:43:31,8


In [20]:
# Répartition des écarts en jours : y a-t-il un pic isolé ?
show(f"""
    SELECT date_diff('day', order_delivered_carrier_date, order_approved_at) AS ecart_jours,
           COUNT(*) AS n
    FROM read_csv_auto('{ORDERS}')
    WHERE order_delivered_carrier_date < order_approved_at
    GROUP BY ecart_jours
    ORDER BY ecart_jours DESC
    LIMIT 40
""")

,ecart_jours,n
0,171,1
1,9,8
2,8,4
3,7,3
4,6,6
5,5,7
6,4,33
7,3,36
8,2,97
9,1,485


In [21]:
# Hypothèse du traitement groupé : approbations autour du 13 septembre 2017
show(f"""
    SELECT CAST(order_approved_at AS DATE) AS jour,
           COUNT(*) AS nb_approbations,
           COUNT(*) FILTER (WHERE order_delivered_carrier_date < order_approved_at) AS avec_transporteur_avant
    FROM read_csv_auto('{ORDERS}')
    WHERE CAST(order_approved_at AS DATE) BETWEEN '2017-09-10' AND '2017-09-16'
    GROUP BY jour
    ORDER BY jour
""")

,jour,nb_approbations,avec_transporteur_avant
0,2017-09-10,106,0
1,2017-09-11,155,0
2,2017-09-12,222,0
3,2017-09-13,213,12
4,2017-09-14,190,0
5,2017-09-15,157,0
6,2017-09-16,125,0


In [22]:
# La commande avec le plus grand écart : toutes ses dates
show(f"""
    SELECT *
    FROM read_csv_auto('{ORDERS}')
    WHERE order_delivered_carrier_date < order_approved_at
    ORDER BY date_diff('day', order_delivered_carrier_date, order_approved_at) DESC
    LIMIT 1
""")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,7c48bb55e8e4f7e56d412e9653db37bc,34ef6181341eb36c47fd601c46878f00,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,2018-08-07


In [69]:
VIOLATIONS = f"""
    SELECT order_id, 'achat_manquant' AS regle
    FROM read_csv_auto('{ORDERS}')
    WHERE order_purchase_timestamp IS NULL
    UNION ALL
    SELECT order_id, 'approbation_manquante'
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status IN ('approved', 'invoiced', 'processing', 'shipped', 'delivered')
      AND order_approved_at IS NULL
    UNION ALL
    SELECT order_id, 'transporteur_manquant'
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status IN ('shipped', 'delivered')
      AND order_delivered_carrier_date IS NULL
    UNION ALL
    SELECT order_id, 'livraison_manquante'
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
    UNION ALL
    SELECT order_id, 'approbation_presente_mais_created'
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status = 'created' AND order_approved_at IS NOT NULL
    UNION ALL
    SELECT order_id, 'transporteur_present_trop_tot'
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status IN ('created', 'approved', 'invoiced', 'processing')
      AND order_delivered_carrier_date IS NOT NULL
    UNION ALL
    SELECT order_id, 'livraison_presente_mais_pas_delivered'
    FROM read_csv_auto('{ORDERS}')
    WHERE order_status <> 'delivered' AND order_delivered_customer_date IS NOT NULL
"""

show(f"SELECT regle, COUNT(*) AS nb_lignes FROM ({VIOLATIONS}) GROUP BY regle ORDER BY nb_lignes DESC")

,regle,nb_lignes
0,approbation_manquante,14
1,livraison_manquante,8
2,livraison_presente_mais_pas_delivered,6
3,transporteur_manquant,2


In [70]:
REGLE = "approbation_manquante"   # change ce nom pour voir une autre règle

show(f"""
    SELECT v.regle, o.order_id, o.order_status, o.order_purchase_timestamp,
           o.order_approved_at, o.order_delivered_carrier_date,
           o.order_delivered_customer_date
    FROM ({VIOLATIONS}) v
    JOIN read_csv_auto('{ORDERS}') o USING (order_id)
    WHERE v.regle = '{REGLE}'
    ORDER BY o.order_purchase_timestamp
    LIMIT 20
""")

,regle,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,approbation_manquante,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01
1,approbation_manquante,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19
2,approbation_manquante,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19
3,approbation_manquante,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,2017-02-22 11:31:30,2017-03-03 11:47:47
4,approbation_manquante,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03
5,approbation_manquante,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23
6,approbation_manquante,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06
7,approbation_manquante,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38
8,approbation_manquante,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33
9,approbation_manquante,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,2017-02-23 03:09:14,2017-03-07 13:57:47


In [71]:
NB_LIGNES = 5   # nombre de lignes à afficher pour chaque règle

TOUTES_LES_REGLES = [
    "achat_manquant",
    "approbation_manquante",
    "transporteur_manquant",
    "livraison_manquante",
    "approbation_presente_mais_created",
    "transporteur_present_trop_tot",
    "livraison_presente_mais_pas_delivered",
]

for regle in TOUTES_LES_REGLES:
    total = con.sql(f"SELECT COUNT(*) FROM ({VIOLATIONS}) WHERE regle = '{regle}'").fetchall()[0][0]
    print(f"\n{'=' * 70}\n{regle} : {total} ligne(s)\n{'=' * 70}")
    if total == 0:
        print("Règle respectée : aucune ligne à examiner.")
        continue
    show(f"""
        SELECT o.order_id, o.order_status, o.order_purchase_timestamp,
               o.order_approved_at, o.order_delivered_carrier_date,
               o.order_delivered_customer_date
        FROM ({VIOLATIONS}) v
        JOIN read_csv_auto('{ORDERS}') o USING (order_id)
        WHERE v.regle = '{regle}'
        ORDER BY o.order_purchase_timestamp
        LIMIT {NB_LIGNES}
    """)


achat_manquant : 0 ligne(s)
Règle respectée : aucune ligne à examiner.

approbation_manquante : 14 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01
1,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19
2,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19
3,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,2017-02-22 11:31:30,2017-03-03 11:47:47
4,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03



transporteur_manquant : 2 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT
1,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47



livraison_manquante : 8 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT
1,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT
2,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT
3,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT
4,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT



approbation_presente_mais_created : 0 ligne(s)
Règle respectée : aucune ligne à examiner.

transporteur_present_trop_tot : 0 ligne(s)
Règle respectée : aucune ligne à examiner.

livraison_presente_mais_pas_delivered : 6 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
0,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34
1,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11
2,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43
3,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59
4,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50


In [72]:
def examiner(violations, regles, nb_lignes=5):
    """Pour chaque règle : le nombre de lignes concernées, puis un aperçu avec toutes les dates."""
    for regle in regles:
        total = con.sql(f"SELECT COUNT(*) FROM ({violations}) WHERE regle = '{regle}'").fetchall()[0][0]
        print(f"\n{'=' * 70}\n{regle} : {total} ligne(s)\n{'=' * 70}")
        if total == 0:
            print("Règle respectée : aucune ligne à examiner.")
            continue
        show(f"""
            SELECT o.order_id, o.order_status, o.order_purchase_timestamp,
                   o.order_approved_at, o.order_delivered_carrier_date,
                   o.order_delivered_customer_date, o.order_estimated_delivery_date
            FROM ({violations}) v
            JOIN read_csv_auto('{ORDERS}') o USING (order_id)
            WHERE v.regle = '{regle}'
            ORDER BY o.order_purchase_timestamp
            LIMIT {nb_lignes}
        """)

In [73]:
REGLES_DATES = {
    "approbation_avant_achat":        "order_approved_at < order_purchase_timestamp",
    "transporteur_avant_achat":       "order_delivered_carrier_date < order_purchase_timestamp",
    "transporteur_avant_approbation": "order_delivered_carrier_date < order_approved_at",
    "livraison_avant_achat":          "order_delivered_customer_date < order_purchase_timestamp",
    "livraison_avant_approbation":    "order_delivered_customer_date < order_approved_at",
    "livraison_avant_transporteur":   "order_delivered_customer_date < order_delivered_carrier_date",
    "estimation_avant_achat":         "order_estimated_delivery_date < order_purchase_timestamp",
}

VIOLATIONS_DATES = " UNION ALL ".join(
    f"SELECT order_id, '{nom}' AS regle FROM read_csv_auto('{ORDERS}') WHERE {condition}"
    for nom, condition in REGLES_DATES.items()
)

# 1. Le résumé : combien de lignes par règle
show(f"SELECT regle, COUNT(*) AS nb_lignes FROM ({VIOLATIONS_DATES}) GROUP BY regle ORDER BY nb_lignes DESC")

# 2. Le détail : les lignes de chaque règle
examiner(VIOLATIONS_DATES, list(REGLES_DATES.keys()), nb_lignes=5)

,regle,nb_lignes
0,transporteur_avant_approbation,1359
1,transporteur_avant_achat,166
2,livraison_avant_approbation,61
3,livraison_avant_transporteur,23



approbation_avant_achat : 0 ligne(s)
Règle respectée : aucune ligne à examiner.

transporteur_avant_achat : 166 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,fd58bdbb6cc5d7519e063c3ceb7e951b,delivered,2018-04-25 13:02:26,2018-04-25 13:15:26,2018-04-25 12:36:00,2018-05-08 16:32:17,2018-05-28
1,b4180badf3ac49892e23d6e0276f1e84,delivered,2018-04-26 14:44:05,2018-04-26 14:55:14,2018-04-26 13:51:00,2018-05-03 00:14:44,2018-06-06
2,e4c540c341b9a50b1a1b28e71867eb5b,delivered,2018-04-26 15:55:42,2018-04-26 16:11:01,2018-04-26 14:55:00,2018-04-27 17:24:52,2018-05-09
3,9e6928e55ec6948338048d72ebdf2305,delivered,2018-04-27 15:12:36,2018-04-27 15:32:59,2018-04-27 15:08:00,2018-05-07 19:03:52,2018-05-29
4,5a339130640419f9e00ab4b6ff61dc22,delivered,2018-04-27 15:59:23,2018-04-27 16:15:25,2018-04-27 14:35:00,2018-05-02 15:32:39,2018-05-16



transporteur_avant_approbation : 1359 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,6b80bb20190715d71c43efff617bd659,delivered,2017-02-19 01:15:03,2017-03-01 10:51:46,2017-02-22 16:05:29,2017-02-24 14:27:26,2017-03-17
1,af2adc7e31b52bdfe068cf60426b54b2,delivered,2017-04-24 09:14:06,2017-04-27 10:32:39,2017-04-26 18:47:58,2017-05-02 17:24:56,2017-05-11
2,1bab77281b20f4719d0a9f87f2282fde,delivered,2017-04-24 22:25:28,2017-04-27 10:33:13,2017-04-27 07:42:03,2017-05-08 16:07:05,2017-05-11
3,69a236fbbc4a603ebfa4468a3bdcb140,delivered,2017-04-25 01:46:02,2017-04-27 10:32:00,2017-04-26 09:11:44,2017-05-03 13:39:47,2017-05-25
4,a60d571514c73cfc7655d67c75ed82c7,delivered,2017-04-25 09:50:01,2017-04-27 10:32:03,2017-04-27 07:18:06,2017-05-05 18:58:07,2017-05-26



livraison_avant_achat : 0 ligne(s)
Règle respectée : aucune ligne à examiner.

livraison_avant_approbation : 61 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,6b80bb20190715d71c43efff617bd659,delivered,2017-02-19 01:15:03,2017-03-01 10:51:46,2017-02-22 16:05:29,2017-02-24 14:27:26,2017-03-17
1,f6f0b2497c5a4ca89670186757ab2684,delivered,2017-06-26 12:23:32,2017-07-05 17:09:11,2017-06-27 17:03:09,2017-07-03 17:53:29,2017-07-18
2,70f357cca87c1162357bf3c0a993cbe5,delivered,2017-09-01 18:40:11,2017-09-13 22:04:49,2017-09-05 12:33:19,2017-09-12 16:53:41,2017-09-22
3,8554cb37f7158cb0b082a841d24a4589,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,2017-09-08 20:07:45,2017-10-02
4,cf72398d0690f841271b695bbfda82d2,delivered,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,2017-09-11 14:15:02,2017-09-21



livraison_avant_transporteur : 23 ligne(s)


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,76458889992169d3135b264dc13aec67,delivered,2016-10-07 10:05:16,2016-10-07 11:24:43,2016-10-26 11:43:06,2016-10-20 18:03:17,2016-11-29
1,f688669f48063536e082bb32d634cd46,delivered,2016-10-07 10:28:56,2016-10-11 04:56:13,2016-10-21 18:02:40,2016-10-20 20:33:27,2016-11-29
2,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-07 17:09:56,2016-10-07 17:32:09,2016-10-26 11:42:05,2016-10-20 19:07:54,2016-12-01
3,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
4,b866af202be0692766081310cd4085e1,delivered,2017-01-27 14:59:17,2017-01-27 15:30:46,2017-02-20 02:32:08,2017-02-15 03:53:46,2017-04-17



estimation_avant_achat : 0 ligne(s)
Règle respectée : aucune ligne à examiner.


In [74]:
show(f"""
    SELECT order_id, order_status, order_purchase_timestamp, order_delivered_customer_date,
           date_diff('day', order_purchase_timestamp, order_delivered_customer_date) AS duree_jours
    FROM read_csv_auto('{ORDERS}')
    WHERE order_delivered_customer_date IS NOT NULL
    ORDER BY duree_jours DESC
    LIMIT 10
""")

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,duree_jours
0,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,210
1,1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23 14:57:35,2018-09-19 23:24:07,208
2,440d0d17af552815d15a9e41abe49359,delivered,2017-03-07 23:59:51,2017-09-19 15:12:50,196
3,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,2017-03-08 18:09:02,2017-09-19 14:33:17,195
4,285ab9426d6982034523a855f55a885e,delivered,2017-03-08 22:47:40,2017-09-19 14:00:04,195
5,0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09 13:26:57,2017-09-19 14:38:21,194
6,47b40429ed8cce3aee9199792275433f,delivered,2018-01-03 09:44:01,2018-07-13 20:51:31,191
7,2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13 20:17:10,2017-09-19 17:00:07,190
8,c27815f7e3dd0b926b58552628481575,delivered,2017-03-15 23:23:17,2017-09-19 17:14:25,188
9,2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15 11:24:27,2017-09-19 14:38:18,188


### Conclusions sur `orders` (à remplir avec tes mots)

- **Une ligne =** 
- **Clé :** 
- **Les dates vides normales sont celles des statuts :** 
- **Les anomalies trouvées (avec les nombres) :** 
- **Ce que je ferai en Silver pour chacune (garder et marquer, ou quarantaine) :** 

## 2. `olist_customers_dataset.csv` : les clients

**À retenir :** une ligne = **un client pour une commande**. Clé : `customer_id`.

**Le piège :** `customer_id` change à chaque commande. Le vrai identifiant d'une personne est `customer_unique_id`.

In [23]:
show(f"SELECT * FROM read_csv_auto('{CUSTOMERS}') LIMIT 5")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [24]:
profil(CUSTOMERS)

99441 lignes, 5 colonnes


,colonne,type,vides,distincts,pct_vides
0,customer_id,VARCHAR,0,99441,0.0
1,customer_unique_id,VARCHAR,0,96096,0.0
2,customer_zip_code_prefix,VARCHAR,0,14994,0.0
3,customer_city,VARCHAR,0,4119,0.0
4,customer_state,VARCHAR,0,27,0.0


In [25]:
show(f"""
    SELECT COUNT(*) AS lignes,
           COUNT(DISTINCT customer_id) AS customer_id_distincts,
           COUNT(DISTINCT customer_unique_id) AS clients_reels
    FROM read_csv_auto('{CUSTOMERS}')
""")

,lignes,customer_id_distincts,clients_reels
0,99441,99441,96096


### Combien de commandes par vrai client ?

In [26]:
show(f"""
    SELECT nb_commandes, COUNT(*) AS nb_clients
    FROM (
        SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS nb_commandes
        FROM read_csv_auto('{ORDERS}') o
        JOIN read_csv_auto('{CUSTOMERS}') c USING (customer_id)
        GROUP BY c.customer_unique_id
    )
    GROUP BY nb_commandes
    ORDER BY nb_commandes
""")

,nb_commandes,nb_clients
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


### Où habitent les clients ? (les 10 États les plus fréquents)

In [27]:
show(f"""
    SELECT customer_state, COUNT(*) AS lignes, COUNT(DISTINCT customer_unique_id) AS clients_reels
    FROM read_csv_auto('{CUSTOMERS}')
    GROUP BY customer_state
    ORDER BY lignes DESC
    LIMIT 10
""")

,customer_state,lignes,clients_reels
0,SP,41746,40302
1,RJ,12852,12384
2,MG,11635,11259
3,RS,5466,5277
4,PR,5045,4882
5,SC,3637,3534
6,BA,3380,3277
7,DF,2140,2075
8,ES,2033,1964
9,GO,2020,1952


NBre de commandes par customer : pour tous les customers on a chaque commande correspond a un customer id 

In [5]:
show(f"""
    SELECT nb_commandes, COUNT(*) AS nb_customer_id
    FROM (
        SELECT customer_id, COUNT(*) AS nb_commandes
        FROM read_csv_auto('{ORDERS}')
        GROUP BY customer_id
    )
    GROUP BY nb_commandes
    ORDER BY nb_commandes
""")

,nb_commandes,nb_customer_id
0,1,99441


In [6]:
show(f"""
    WITH par_personne AS (
        SELECT c.customer_unique_id,
               COUNT(DISTINCT c.customer_id) AS nb_customer_id,
               COUNT(DISTINCT o.order_id) AS nb_commandes
        FROM read_csv_auto('{CUSTOMERS}') c
        LEFT JOIN read_csv_auto('{ORDERS}') o USING (customer_id)
        GROUP BY c.customer_unique_id
    )
    SELECT COUNT(*) AS personnes,
           COUNT(*) FILTER (WHERE nb_customer_id = nb_commandes) AS customer_id_egal_commandes,
           COUNT(*) FILTER (WHERE nb_customer_id <> nb_commandes) AS difference
    FROM par_personne
""")

,personnes,customer_id_egal_commandes,difference
0,96096,96096,0


In [7]:
con.sql(f"""
    CREATE OR REPLACE TABLE orders_avec_personne AS
    SELECT o.*, c.customer_unique_id
    FROM read_csv_auto('{ORDERS}') o
    LEFT JOIN read_csv_auto('{CUSTOMERS}') c USING (customer_id)
""")

show("""
    SELECT COUNT(*) AS lignes,
           COUNT(DISTINCT order_id) AS commandes_distinctes,
           COUNT(*) FILTER (WHERE customer_unique_id IS NULL) AS sans_personne,
           COUNT(DISTINCT customer_unique_id) AS personnes
    FROM orders_avec_personne
""")

,lignes,commandes_distinctes,sans_personne,personnes
0,99441,99441,0,96096


In [8]:
show(f"""
    SELECT COUNT(*) AS personnes_a_plusieurs_lignes,
           COUNT(*) FILTER (WHERE nb_cp > 1 OR nb_villes > 1 OR nb_etats > 1) AS personnes_avec_adresse_variable
    FROM (
        SELECT customer_unique_id,
               COUNT(DISTINCT customer_zip_code_prefix) AS nb_cp,
               COUNT(DISTINCT customer_city) AS nb_villes,
               COUNT(DISTINCT customer_state) AS nb_etats
        FROM read_csv_auto('{CUSTOMERS}')
        GROUP BY customer_unique_id
        HAVING COUNT(*) > 1
    )
""")

,personnes_a_plusieurs_lignes,personnes_avec_adresse_variable
0,2997,252


In [9]:
show(f"""
    WITH multi AS (
        SELECT customer_unique_id,
               COUNT(DISTINCT customer_zip_code_prefix) AS nb_cp,
               COUNT(DISTINCT LOWER(TRIM(customer_city))) AS nb_villes,
               COUNT(DISTINCT customer_state) AS nb_etats
        FROM read_csv_auto('{CUSTOMERS}')
        GROUP BY customer_unique_id
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS personnes_a_plusieurs_lignes,
           COUNT(*) FILTER (WHERE nb_cp > 1 OR nb_villes > 1 OR nb_etats > 1) AS adresse_variable,
           COUNT(*) FILTER (WHERE nb_cp > 1 AND nb_villes = 1 AND nb_etats = 1) AS seul_le_code_postal_change,
           COUNT(*) FILTER (WHERE nb_villes > 1 AND nb_etats = 1) AS ville_change_meme_etat,
           COUNT(*) FILTER (WHERE nb_etats > 1) AS etat_change
    FROM multi
""")

,personnes_a_plusieurs_lignes,adresse_variable,seul_le_code_postal_change,ville_change_meme_etat,etat_change
0,2997,252,130,83,39


In [10]:
show(f"""
    SELECT COUNT(DISTINCT customer_city) AS villes_distinctes,
           COUNT(DISTINCT LOWER(TRIM(customer_city))) AS villes_apres_normalisation
    FROM read_csv_auto('{CUSTOMERS}')
""")

,villes_distinctes,villes_apres_normalisation
0,4119,4119


### Conclusions sur `customers` (à remplir)

- **Une ligne =** 
- **Différence entre `customer_id` et `customer_unique_id` :** 
- **Nombre de vrais clients, et part de ceux qui ont commandé une seule fois :** 
- **Impact pour le dashboard :** 

## 3. `olist_order_items_dataset.csv` : les articles d'une commande

**À retenir :** une ligne = **un article dans une commande**. Clé : `order_id` + `order_item_id` (le numéro de l'article dans la commande).

C'est ici que se trouvent les **montants** (`price`, `freight_value`) et les liens vers `products` et `sellers`.

In [28]:
show(f"SELECT * FROM read_csv_auto('{ITEMS}') LIMIT 5")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [29]:
profil(ITEMS)

112650 lignes, 7 colonnes


,colonne,type,vides,distincts,pct_vides
0,order_id,VARCHAR,0,98666,0.0
1,order_item_id,BIGINT,0,21,0.0
2,product_id,VARCHAR,0,32951,0.0
3,seller_id,VARCHAR,0,3095,0.0
4,shipping_limit_date,TIMESTAMP,0,93318,0.0
5,price,DOUBLE,0,5968,0.0
6,freight_value,DOUBLE,0,6999,0.0


### Le grain : combien d'articles par commande ?

In [30]:
show(f"""
    SELECT COUNT(*) AS lignes_articles,
           COUNT(DISTINCT order_id) AS commandes_distinctes,
           ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT order_id), 2) AS articles_par_commande
    FROM read_csv_auto('{ITEMS}')
""")

,lignes_articles,commandes_distinctes,articles_par_commande
0,112650,98666,1.14


In [31]:
show(f"""
    SELECT nb_articles, COUNT(*) AS nb_commandes
    FROM (
        SELECT order_id, COUNT(*) AS nb_articles
        FROM read_csv_auto('{ITEMS}')
        GROUP BY order_id
    )
    GROUP BY nb_articles
    ORDER BY nb_articles
    LIMIT 15
""")

,nb_articles,nb_commandes
0,1,88863
1,2,7516
2,3,1322
3,4,505
4,5,204
5,6,198
6,7,22
7,8,8
8,9,3
9,10,8


### Les montants : prix et frais de port

In [32]:
show(f"""
    SELECT MIN(price) AS prix_min,
           MAX(price) AS prix_max,
           ROUND(AVG(price), 2) AS prix_moyen,
           MEDIAN(price) AS prix_median,
           COUNT(*) FILTER (WHERE price <= 0) AS prix_nuls_ou_negatifs,
           MIN(freight_value) AS port_min,
           MAX(freight_value) AS port_max,
           COUNT(*) FILTER (WHERE freight_value = 0) AS port_a_zero,
           COUNT(*) FILTER (WHERE freight_value > price) AS port_superieur_au_prix
    FROM read_csv_auto('{ITEMS}')
""")

,prix_min,prix_max,prix_moyen,prix_median,prix_nuls_ou_negatifs,port_min,port_max,port_a_zero,port_superieur_au_prix
0,0.85,6735.0,120.65,74.99,0,0.0,409.68,383,4124


In [33]:
# Les 5 articles les plus chers : valeurs extrêmes ou normales ?
show(f"""
    SELECT order_id, order_item_id, product_id, price, freight_value
    FROM read_csv_auto('{ITEMS}')
    ORDER BY price DESC
    LIMIT 5
""")

,order_id,order_item_id,product_id,price,freight_value
0,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,6735.0,194.31
1,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,6729.0,193.21
2,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,6499.0,227.66
3,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,4799.0,151.34
4,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,4690.0,74.34


### Conclusions sur `order_items` (à remplir)

- **Une ligne =** 
- **Clé composée :** 
- **Une commande a en moyenne … articles, au maximum … :** 
- **Anomalies sur les montants :** 

## 4. `olist_order_payments_dataset.csv` : les paiements

**À retenir :** une ligne = **un règlement d'une commande**. Clé : `order_id` + `payment_sequential`.

**Le paiement est rattaché à la commande**, pas à un article : il n'y a ni `product_id` ni `order_item_id` dans cette table.

In [34]:
show(f"SELECT * FROM read_csv_auto('{PAYMENTS}') LIMIT 5")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [35]:
profil(PAYMENTS)

103886 lignes, 5 colonnes


,colonne,type,vides,distincts,pct_vides
0,order_id,VARCHAR,0,99440,0.0
1,payment_sequential,BIGINT,0,29,0.0
2,payment_type,VARCHAR,0,5,0.0
3,payment_installments,BIGINT,0,24,0.0
4,payment_value,DOUBLE,0,29077,0.0


### Combien de paiements par commande ?

In [36]:
show(f"""
    SELECT nb_paiements, COUNT(*) AS nb_commandes
    FROM (
        SELECT order_id, COUNT(*) AS nb_paiements
        FROM read_csv_auto('{PAYMENTS}')
        GROUP BY order_id
    )
    GROUP BY nb_paiements
    ORDER BY nb_paiements
""")

,nb_paiements,nb_commandes
0,1,96479
1,2,2382
2,3,301
3,4,108
4,5,52
5,6,36
6,7,28
7,8,11
8,9,9
9,10,5


### Pourquoi plusieurs paiements ? Deux exemples de commandes à 3 paiements

In [37]:
show(f"""
    SELECT order_id, payment_sequential, payment_type, payment_installments, payment_value
    FROM read_csv_auto('{PAYMENTS}')
    WHERE order_id IN (
        SELECT order_id
        FROM read_csv_auto('{PAYMENTS}')
        GROUP BY order_id HAVING COUNT(*) = 3
        LIMIT 2
    )
    ORDER BY order_id, payment_sequential
""")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,398abc38612b5d4dd9f77541c38d06ce,1,credit_card,1,7.87
1,398abc38612b5d4dd9f77541c38d06ce,2,voucher,1,56.70
2,398abc38612b5d4dd9f77541c38d06ce,3,voucher,1,113.41
3,e86b1b2dd48839d7351406434afb578d,1,voucher,1,20.00
4,e86b1b2dd48839d7351406434afb578d,2,voucher,1,10.60
5,e86b1b2dd48839d7351406434afb578d,3,voucher,1,1.78


### Types de paiement et valeurs

In [38]:
show(f"""
    SELECT payment_type,
           COUNT(*) AS n,
           COUNT(*) FILTER (WHERE payment_value = 0) AS valeur_zero,
           MAX(payment_installments) AS echeances_max,
           ROUND(SUM(payment_value), 2) AS montant_total
    FROM read_csv_auto('{PAYMENTS}')
    GROUP BY payment_type
    ORDER BY n DESC
""")

,payment_type,n,valeur_zero,echeances_max,montant_total
0,credit_card,76795,0,24,12542084.19
1,boleto,19784,0,1,2869361.27
2,voucher,5775,6,1,379436.87
3,debit_card,1529,0,1,217989.79
4,not_defined,3,3,1,0.00


### Le montant payé correspond-il au montant des articles ?

On **agrège d'abord** chaque table à une ligne par commande, puis on joint. C'est la bonne méthode pour éviter le piège de la jointure qui duplique les lignes (voir partie 9).

In [39]:
show(f"""
    WITH i AS (
        SELECT order_id, SUM(price + freight_value) AS total_articles
        FROM read_csv_auto('{ITEMS}')
        GROUP BY order_id
    ),
    p AS (
        SELECT order_id, SUM(payment_value) AS total_paye
        FROM read_csv_auto('{PAYMENTS}')
        GROUP BY order_id
    )
    SELECT COUNT(*) AS commandes_comparees,
           COUNT(*) FILTER (WHERE ABS(total_articles - total_paye) < 0.01) AS montants_identiques,
           COUNT(*) FILTER (WHERE ABS(total_articles - total_paye) >= 0.01) AS montants_differents
    FROM i JOIN p USING (order_id)
""")

,commandes_comparees,montants_identiques,montants_differents
0,98665,98285,380


### Conclusions sur `order_payments` (à remplir)

- **Une ligne =** 
- **Le paiement est rattaché à :** 
- **Pourquoi certaines commandes ont plusieurs paiements :** 
- **Le total payé correspond-il aux articles ? Qu'en conclus-tu ?** 

## 5. `olist_order_reviews_dataset.csv` : les avis

**À retenir :** une ligne = **un avis**. Clé attendue : `review_id`. Une commande peut avoir plus d'un avis.

**Ce qu'on cherche :** la distribution des notes, la part des avis commentés, et les doublons.

In [40]:
show(f"SELECT * FROM read_csv_auto('{REVIEWS}') LIMIT 5")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela Internet se...,2018-03-01,2018-03-02 10:26:53


In [41]:
profil(REVIEWS)

99224 lignes, 7 colonnes


,colonne,type,vides,distincts,pct_vides
0,review_id,VARCHAR,0,98410,0.00
1,order_id,VARCHAR,0,98673,0.00
2,review_score,BIGINT,0,5,0.00
3,review_comment_title,VARCHAR,87656,4527,88.34
4,review_comment_message,VARCHAR,58247,36159,58.70
5,review_creation_date,TIMESTAMP,0,636,0.00
6,review_answer_timestamp,TIMESTAMP,0,98248,0.00


In [42]:
show(f"""
    SELECT review_score,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
           COUNT(review_comment_message) AS avec_commentaire
    FROM read_csv_auto('{REVIEWS}')
    GROUP BY review_score
    ORDER BY review_score
""")

,review_score,n,pct,avec_commentaire
0,1,11424,11.51,8745
1,2,3151,3.18,2145
2,3,8179,8.24,3557
3,4,19142,19.29,5976
4,5,57328,57.78,20554


### Les clés sont-elles uniques ? Combien d'avis par commande ?

In [43]:
show(f"""
    SELECT COUNT(*) AS lignes,
           COUNT(DISTINCT review_id) AS review_id_distincts,
           COUNT(DISTINCT order_id) AS commandes_distinctes
    FROM read_csv_auto('{REVIEWS}')
""")

,lignes,review_id_distincts,commandes_distinctes
0,99224,98410,98673


In [44]:
show(f"""
    SELECT nb_avis, COUNT(*) AS nb_commandes
    FROM (
        SELECT order_id, COUNT(*) AS nb_avis
        FROM read_csv_auto('{REVIEWS}')
        GROUP BY order_id
    )
    GROUP BY nb_avis
    ORDER BY nb_avis
""")

,nb_avis,nb_commandes
0,1,98126
1,2,543
2,3,4


### Conclusions sur `order_reviews` (à remplir)

- **Une ligne =** 
- **Note moyenne approximative et note dominante :** 
- **Part des avis avec un commentaire :** 
- **Problèmes de clés (`review_id` en double ?) et décision pour Silver :** 

## 6. `olist_products_dataset.csv` et `product_category_name_translation.csv`

**À retenir :** une ligne de `products` = **un produit**. Clé : `product_id`.
La table de traduction donne le nom de catégorie en anglais : une ligne = une catégorie.

**Ce qu'on cherche :** les catégories vides, celles sans traduction, et les noms de colonnes mal orthographiés dans la source (`product_name_lenght`).

In [45]:
show(f"SELECT * FROM read_csv_auto('{PRODUCTS}') LIMIT 5")

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


In [46]:
profil(PRODUCTS)

32951 lignes, 9 colonnes


,colonne,type,vides,distincts,pct_vides
0,product_id,VARCHAR,0,32951,0.00
1,product_category_name,VARCHAR,610,73,1.85
2,product_name_lenght,BIGINT,610,66,1.85
3,product_description_lenght,BIGINT,610,2960,1.85
4,product_photos_qty,BIGINT,610,19,1.85
5,product_weight_g,BIGINT,2,2204,0.01
6,product_length_cm,BIGINT,2,99,0.01
7,product_height_cm,BIGINT,2,102,0.01
8,product_width_cm,BIGINT,2,95,0.01


In [47]:
show(f"""
    SELECT COUNT(*) AS produits,
           COUNT(*) FILTER (WHERE product_category_name IS NULL) AS categorie_vide,
           COUNT(DISTINCT product_category_name) AS categories_distinctes
    FROM read_csv_auto('{PRODUCTS}')
""")

,produits,categorie_vide,categories_distinctes
0,32951,610,73


### Les catégories qui n'ont pas de traduction anglaise

In [48]:
show(f"""
    SELECT p.product_category_name, COUNT(*) AS nb_produits
    FROM read_csv_auto('{PRODUCTS}') p
    LEFT JOIN read_csv_auto('{CATEG}') t
        ON p.product_category_name = t.product_category_name
    WHERE t.product_category_name IS NULL
      AND p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
    ORDER BY nb_produits DESC
""")

,product_category_name,nb_produits
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


### Les 10 catégories les plus fréquentes

In [49]:
show(f"""
    SELECT p.product_category_name, t.product_category_name_english, COUNT(*) AS nb_produits
    FROM read_csv_auto('{PRODUCTS}') p
    LEFT JOIN read_csv_auto('{CATEG}') t
        ON p.product_category_name = t.product_category_name
    GROUP BY p.product_category_name, t.product_category_name_english
    ORDER BY nb_produits DESC
    LIMIT 10
""")

,product_category_name,product_category_name_english,nb_produits
0,cama_mesa_banho,bed_bath_table,3029
1,esporte_lazer,sports_leisure,2867
2,moveis_decoracao,furniture_decor,2657
3,beleza_saude,health_beauty,2444
4,utilidades_domesticas,housewares,2335
5,automotivo,auto,1900
6,informatica_acessorios,computers_accessories,1639
7,brinquedos,toys,1411
8,relogios_presentes,watches_gifts,1329
9,telefonia,telephony,1134


In [50]:
show(f"SELECT * FROM read_csv_auto('{CATEG}') LIMIT 5")

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


### Conclusions sur `products` (à remplir)

- **Une ligne =** 
- **Produits sans catégorie, et valeur de remplacement choisie pour Silver :** 
- **Catégories sans traduction :** 
- **Colonnes à renommer (fautes de la source) :** 

## 7. `olist_sellers_dataset.csv` : les vendeurs

**À retenir :** une ligne = **un vendeur**. Clé : `seller_id`.

In [51]:
show(f"SELECT * FROM read_csv_auto('{SELLERS}') LIMIT 5")

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [52]:
profil(SELLERS)

3095 lignes, 4 colonnes


,colonne,type,vides,distincts,pct_vides
0,seller_id,VARCHAR,0,3095,0.0
1,seller_zip_code_prefix,VARCHAR,0,2246,0.0
2,seller_city,VARCHAR,0,611,0.0
3,seller_state,VARCHAR,0,23,0.0


In [53]:
show(f"""
    SELECT seller_state, COUNT(*) AS nb_vendeurs
    FROM read_csv_auto('{SELLERS}')
    GROUP BY seller_state
    ORDER BY nb_vendeurs DESC
    LIMIT 10
""")

,seller_state,nb_vendeurs
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129
6,GO,40
7,DF,30
8,ES,23
9,BA,19


### Les ventes sont-elles concentrées chez quelques vendeurs ?

In [54]:
show(f"""
    SELECT seller_id, COUNT(*) AS nb_articles, ROUND(SUM(price), 2) AS chiffre_affaires
    FROM read_csv_auto('{ITEMS}')
    GROUP BY seller_id
    ORDER BY chiffre_affaires DESC
    LIMIT 10
""")

,seller_id,nb_articles,chiffre_affaires
0,4869f7a5dfa277a7dca6462dcf3b52b2,1156,229472.63
1,53243585a1d6dc2643021fd1853d8905,410,222776.05
2,4a3ca9315b744ce9f8e9374361493884,1987,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,586,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,1364,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,340,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,1551,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,1171,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,1428,138968.55
9,955fee9216a65b617aa5c0531780ce60,1499,135171.70


### Conclusions sur `sellers` (à remplir)

- **Une ligne =** 
- **Nombre de vendeurs, et concentration des ventes :** 

## 8. `olist_geolocation_dataset.csv` : la géolocalisation

**À retenir :** cette table contient **plus d'un million de lignes** pour bien moins de codes postaux. Une ligne = **un point de coordonnées** associé à un code postal, et un même code postal apparaît donc plusieurs fois.

**Le risque :** joindre cette table sur le code postal **duplique** les lignes, comme le piège des paiements.

In [55]:
show(f"SELECT * FROM read_csv_auto('{GEO}') LIMIT 5")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


In [56]:
profil(GEO)

1000163 lignes, 5 colonnes


,colonne,type,vides,distincts,pct_vides
0,geolocation_zip_code_prefix,VARCHAR,0,19015,0.0
1,geolocation_lat,DOUBLE,0,717372,0.0
2,geolocation_lng,DOUBLE,0,717615,0.0
3,geolocation_city,VARCHAR,0,8011,0.0
4,geolocation_state,VARCHAR,0,27,0.0


In [57]:
show(f"""
    SELECT COUNT(*) AS lignes,
           COUNT(DISTINCT geolocation_zip_code_prefix) AS codes_postaux_distincts,
           ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT geolocation_zip_code_prefix), 1) AS lignes_par_code_postal
    FROM read_csv_auto('{GEO}')
""")

,lignes,codes_postaux_distincts,lignes_par_code_postal
0,1000163,19015,52.6


In [58]:
# Les codes postaux les plus répétés
show(f"""
    SELECT geolocation_zip_code_prefix, COUNT(*) AS nb_lignes,
           COUNT(DISTINCT geolocation_lat) AS latitudes_distinctes
    FROM read_csv_auto('{GEO}')
    GROUP BY geolocation_zip_code_prefix
    ORDER BY nb_lignes DESC
    LIMIT 5
""")

,geolocation_zip_code_prefix,nb_lignes,latitudes_distinctes
0,24220,1146,410
1,24230,1102,327
2,38400,965,746
3,35500,907,726
4,11680,879,726


### Des coordonnées en dehors du Brésil ?

In [59]:
show(f"""
    SELECT COUNT(*) AS hors_bresil
    FROM read_csv_auto('{GEO}')
    WHERE geolocation_lat NOT BETWEEN -34 AND 6
       OR geolocation_lng NOT BETWEEN -74 AND -34
""")

,hors_bresil
0,42


### Conclusions sur `geolocation` (à remplir)

- **Une ligne =** 
- **Pourquoi cette table ne peut pas être jointe directement :** 
- **Ce que je ferai en Silver (une ligne par code postal, moyenne des coordonnées ?) :** 

## 9. Les relations entre les tables

```
customers ──(customer_id)── orders ──(order_id)──┬── order_items ──(product_id)── products ──(category)── category_translation
                                                  │                └─(seller_id)── sellers
                                                  ├── order_payments
                                                  └── order_reviews

geolocation : liée aux clients et aux vendeurs par le code postal
```

### 9.1 Vue d'ensemble des 9 fichiers

In [60]:
lignes = []
for p in sorted(RAW.glob("*.csv")):
    path = p.as_posix()
    n = con.sql(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchall()[0][0]
    nb_col = len(con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").fetchall())
    lignes.append({"fichier": p.name, "lignes": n, "colonnes": nb_col})
display(pd.DataFrame(lignes))

,fichier,lignes,colonnes
0,olist_customers_dataset.csv,99441,5
1,olist_geolocation_dataset.csv,1000163,5
2,olist_order_items_dataset.csv,112650,7
3,olist_order_payments_dataset.csv,103886,5
4,olist_order_reviews_dataset.csv,99224,7
5,olist_orders_dataset.csv,99441,8
6,olist_products_dataset.csv,32951,9
7,olist_sellers_dataset.csv,3095,4
8,product_category_name_translation.csv,71,2


### 9.2 Le grain : lignes contre commandes distinctes

In [61]:
resultats = []
for nom, path in [("orders", ORDERS), ("order_items", ITEMS),
                  ("order_payments", PAYMENTS), ("order_reviews", REVIEWS)]:
    r = con.sql(f"SELECT COUNT(*), COUNT(DISTINCT order_id) FROM read_csv_auto('{path}')").fetchall()[0]
    resultats.append({"table": nom, "lignes": r[0], "commandes_distinctes": r[1]})
display(pd.DataFrame(resultats))

,table,lignes,commandes_distinctes
0,orders,99441,99441
1,order_items,112650,98666
2,order_payments,103886,99440
3,order_reviews,99224,98673


### 9.3 Le piège de la jointure

`order_items` et `order_payments` ont toutes les deux **plusieurs lignes par commande**. Si on les joint directement, chaque article est répété pour chaque paiement, et le chiffre d'affaires gonfle.

In [62]:
show(f"""
    WITH items AS (SELECT * FROM read_csv_auto('{ITEMS}')),
         payments AS (SELECT * FROM read_csv_auto('{PAYMENTS}'))
    SELECT
        (SELECT SUM(price) FROM items) AS ca_direct,
        (SELECT SUM(i.price) FROM items i JOIN payments p ON i.order_id = p.order_id) AS ca_apres_jointure
""")

,ca_direct,ca_apres_jointure
0,1.359164e+07,1.420912e+07


In [63]:
# Nombre de paiements et d'articles par commande : où se situe le risque de duplication ?
show(f"""
    WITH paiements AS (
        SELECT order_id, COUNT(*) AS nb_paiements
        FROM read_csv_auto('{PAYMENTS}') GROUP BY order_id
    ),
    articles AS (
        SELECT order_id, COUNT(*) AS nb_articles
        FROM read_csv_auto('{ITEMS}') GROUP BY order_id
    )
    SELECT p.nb_paiements, a.nb_articles, COUNT(*) AS nb_commandes
    FROM paiements p JOIN articles a ON p.order_id = a.order_id
    GROUP BY p.nb_paiements, a.nb_articles
    ORDER BY p.nb_paiements, a.nb_articles
    LIMIT 20
""")

,nb_paiements,nb_articles,nb_commandes
0,1,1,86202
1,1,2,7288
2,1,3,1290
3,1,4,495
4,1,5,201
5,1,6,196
6,1,7,21
7,1,8,8
8,1,9,3
9,1,10,8


In [64]:
# Les commandes avec le plus d'articles
show(f"""
    WITH articles AS (
        SELECT order_id, COUNT(*) AS nb_articles
        FROM read_csv_auto('{ITEMS}') GROUP BY order_id
    ),
    paiements AS (
        SELECT order_id, COUNT(*) AS nb_paiements
        FROM read_csv_auto('{PAYMENTS}') GROUP BY order_id
    )
    SELECT a.order_id, a.nb_articles, p.nb_paiements
    FROM articles a JOIN paiements p USING (order_id)
    ORDER BY a.nb_articles DESC, p.nb_paiements DESC
    LIMIT 10
""")

,order_id,nb_articles,nb_paiements
0,8272b63d03f5f79c56e9e4120aec44ef,21,1
1,ab14fdcfbe524636d65ee38360e22ce8,20,1
2,1b15974a0141d54e36626dca3fdc731a,20,1
3,9ef13efd6949e4573a18964dd1bbe7f5,15,1
4,428a2f660dc84138d969ccd69a0ab6d5,15,1
5,73c8ab38f07dc94389065f7eba4f297a,14,1
6,9bdc4d4c71aa1de4606060929dee888c,14,1
7,37ee401157a3a0b28c9c6d0ed8c3b24b,13,1
8,3a213fcdfe7d98be74ea0dc05a8b31ae,12,1
9,2c2a19b5703863c908512d135aa6accc,12,1


**À toi :** pourquoi les deux chiffres (`ca_direct` et `ca_apres_jointure`) ne sont-ils pas égaux ? Dans quel cas précis une commande est-elle comptée en double ?

### 9.4 Les commandes sans article

In [65]:
show(f"""
    SELECT o.order_status, COUNT(*) AS n
    FROM read_csv_auto('{ORDERS}') o
    LEFT JOIN read_csv_auto('{ITEMS}') i USING (order_id)
    WHERE i.order_id IS NULL
    GROUP BY o.order_status
    ORDER BY n DESC
""")

,order_status,n
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [66]:
# Parmi elles : celles dont le statut est incohérent (ni annulée ni indisponible)
show(f"""
    SELECT o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at
    FROM read_csv_auto('{ORDERS}') o
    LEFT JOIN read_csv_auto('{ITEMS}') i USING (order_id)
    WHERE i.order_id IS NULL
      AND o.order_status IN ('created', 'invoiced', 'shipped')
""")

,order_id,order_status,order_purchase_timestamp,order_approved_at
0,7a4df5d8cff4090e541401a20a22bb80,created,2017-11-25 11:10:33,NaT
1,b5359909123fa03c50bdb0cfed07f098,created,2017-12-05 01:07:52,NaT
2,dba5062fbda3af4fb6c33b1e040ca38f,created,2018-02-09 17:21:04,NaT
3,35de4050331c6c644cddc86f4f2d0d64,created,2017-12-05 01:07:58,NaT
4,2ce9683175cdab7d1c95bcbb3e36f478,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39
5,e04f1da1f48bf2bbffcf57b9824f76e1,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38
6,90ab3e7d52544ec7bc3363c82689965f,created,2017-11-06 13:12:34,NaT
7,a68ce1686d536ca72bd2dadc4b8671e5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22


### 9.5 Les clés sont-elles uniques ?

In [67]:
controles = [
    ("orders", ORDERS, "order_id"),
    ("customers", CUSTOMERS, "customer_id"),
    ("products", PRODUCTS, "product_id"),
    ("sellers", SELLERS, "seller_id"),
    ("order_reviews", REVIEWS, "review_id"),
    ("order_items", ITEMS, "order_id, order_item_id"),
    ("order_payments", PAYMENTS, "order_id, payment_sequential"),
]
resultats = []
for nom, path, cle in controles:
    lignes, doublons = con.sql(f"""
        SELECT COUNT(*),
               (SELECT COUNT(*) FROM (
                    SELECT {cle} FROM read_csv_auto('{path}')
                    GROUP BY {cle} HAVING COUNT(*) > 1))
        FROM read_csv_auto('{path}')
    """).fetchall()[0]
    resultats.append({"table": nom, "cle": cle, "lignes": lignes, "cles_en_double": doublons})
display(pd.DataFrame(resultats))

,table,cle,lignes,cles_en_double
0,orders,order_id,99441,0
1,customers,customer_id,99441,0
2,products,product_id,32951,0
3,sellers,seller_id,3095,0
4,order_reviews,review_id,99224,789
5,order_items,"order_id, order_item_id",112650,0
6,order_payments,"order_id, payment_sequential",103886,0


### 9.6 Y a-t-il des lignes « orphelines » (qui pointent vers un élément inexistant) ?

In [68]:
show(f"""
    WITH o AS (SELECT * FROM read_csv_auto('{ORDERS}')),
         c AS (SELECT * FROM read_csv_auto('{CUSTOMERS}')),
         i AS (SELECT * FROM read_csv_auto('{ITEMS}')),
         p AS (SELECT * FROM read_csv_auto('{PRODUCTS}')),
         s AS (SELECT * FROM read_csv_auto('{SELLERS}')),
         pay AS (SELECT * FROM read_csv_auto('{PAYMENTS}')),
         r AS (SELECT * FROM read_csv_auto('{REVIEWS}'))
    SELECT
        (SELECT COUNT(*) FROM o WHERE NOT EXISTS (SELECT 1 FROM c WHERE c.customer_id = o.customer_id)) AS commandes_sans_client,
        (SELECT COUNT(*) FROM i WHERE NOT EXISTS (SELECT 1 FROM o WHERE o.order_id = i.order_id)) AS articles_sans_commande,
        (SELECT COUNT(*) FROM i WHERE NOT EXISTS (SELECT 1 FROM p WHERE p.product_id = i.product_id)) AS articles_sans_produit,
        (SELECT COUNT(*) FROM i WHERE NOT EXISTS (SELECT 1 FROM s WHERE s.seller_id = i.seller_id)) AS articles_sans_vendeur,
        (SELECT COUNT(*) FROM pay WHERE NOT EXISTS (SELECT 1 FROM o WHERE o.order_id = pay.order_id)) AS paiements_sans_commande,
        (SELECT COUNT(*) FROM r WHERE NOT EXISTS (SELECT 1 FROM o WHERE o.order_id = r.order_id)) AS avis_sans_commande,
        (SELECT COUNT(*) FROM o WHERE NOT EXISTS (SELECT 1 FROM pay WHERE pay.order_id = o.order_id)) AS commandes_sans_paiement
""")

,commandes_sans_client,articles_sans_commande,articles_sans_produit,articles_sans_vendeur,paiements_sans_commande,avis_sans_commande,commandes_sans_paiement
0,0,0,0,0,0,0,1


## 10. Synthèse : les règles de qualité pour Silver

Recopie ici, avec les **nombres que tu as trouvés**, toutes les anomalies. Ce tableau devient la liste de tes règles de quarantaine et de tes tests dbt.

| # | Table | Règle violée | Nombre de lignes | Décision pour Silver (quarantaine, marquer, corriger) |
|---|---|---|---|---|
| 1 | orders | `delivered` sans date de livraison | | |
| 2 | orders | `delivered` sans date d'approbation | | |
| 3 | orders | `delivered` sans date de remise au transporteur | | |
| 4 | orders | `canceled` avec une date de livraison | | |
| 5 | orders | remise au transporteur avant l'approbation | | |
| 6 | orders | livraison avant la remise au transporteur | | |
| 7 | orders | sans article, statut `created` / `invoiced` / `shipped` | | |
| 8 | products | catégorie vide | | |
| 9 | products | catégorie sans traduction | | |
| 10 | order_reviews | clé en double ? | | |
| 11 | geolocation | plusieurs lignes par code postal | | |

### Ma carte des relations (à dessiner toi-même)

Dessine sur papier ou dans un outil de schéma : `orders` au centre, les autres tables autour, avec la colonne qui les relie et le grain de chacune. Ajoute-la ensuite à `docs/data_dictionary.md`.

### Questions de synthèse (réponds avec tes mots)

1. Quelle est la table centrale du modèle, et pourquoi ?
2. Pourquoi le chiffre d'affaires se gonfle-t-il quand on joint `order_items` et `order_payments` ?
3. Quelle est la différence entre une commande, un article et un paiement ?
4. Pourquoi ne supprime-t-on pas les lignes anormales en silence ?